In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.8460000000000004, 10: 0.8345, 20: 0.8400000000000001, 30: 0.8395000000000001, 40: 0.8414999999999999, 50: 0.8385000000000004, 60: 0.8420000000000002, 70: 0.8475000000000001, 80: 0.8484999999999999, 90: 0.8505, 100: 0.8475000000000001, 110: 0.8565000000000002, 120: 0.8555000000000001, 130: 0.8564999999999999, 140: 0.8510000000000002, 150: 0.8565000000000003, 160: 0.8550000000000002, 170: 0.8525000000000003, 180: 0.8625000000000002, 190: 0.8640000000000002, 200: 0.8595000000000003, 210: 0.8540000000000001, 220: 0.8555000000000001, 230: 0.8605000000000003, 240: 0.8575000000000003, 250: 0.8560000000000001, 260: 0.8574999999999999, 270: 0.8535, 280: 0.8584210526315789, 290: 0.8552631578947366, 300: 0.8573684210526316}
{0: 0.0027639999999999995, 10: 0.00249975, 20: 0.002480000000000001, 30: 0.0021497499999999998, 40: 0.00204775, 50: 0.0030277499999999996, 60: 0.002956, 70: 0.0022337499999999996, 80: 0.0025977500000000007, 90: 0.00265975, 100: 0.0034537500000000007, 110: 0.0017977500000